# Leader-Follower Unified Sweep v2

Single-source Table 7: CI, CD, and true DLinear in one pipeline, one CSV.

**Experiment:** C=21, rho=0.5, phi=0.8, gamma in {0.0, 0.3, 0.6, 0.9},
seeds {42, 123, 456, 789, 1011}, modes {CI, CD, DLinear}. 60 runs total.

**DLinear:** full Zeng et al. (2023) decomposition — MA trend (kernel=25) +
remainder, two independent Linear(512, 96) branches, summed. Identical to
train_dlinear.ipynb.

**Output:** `/kaggle/working/results_leader_follower.csv`
Schema matches the committed results_leader_follower.csv exactly.

**Estimated runtime:** ~9h on T4 (dominated by 20 CD runs at ~800s each).
Checkpoints after every run; safe to interrupt and resume.


In [ ]:
# ── Imports and device setup ──────────────────────────────────────────────────
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ── Experiment configuration ──────────────────────────────────────────────────
# Change SEEDS here to add or remove seeds; everything else follows.

PHI:   float = 0.8
RHO:   float = 0.5
C:     int   = 21         # N_LEADERS + N_FOLLOWERS + N_ISOLATE

GAMMAS: list[float] = [0.0, 0.3, 0.6, 0.9]
SEEDS:  list[int]   = [42, 123, 456, 789, 1011]
MODES:  list[str]   = ["CI", "CD", "DLinear"]

# Sequence (must match results_leader_follower_v3.csv exactly)
SEQ_LEN:      int = 512
PRED_LEN:     int = 96
PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8

# Architecture (Table 2, Synthetic + ETTh1 config)
D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

# Training
LR:             float = 1e-4
WEIGHT_DECAY:   float = 1e-4
MAX_EPOCHS:     int   = 50
PATIENCE:       int   = 10
GRAD_CLIP:      float = 1.0
WARMUP_EPOCHS:  int   = 10   # CI and CD only; DLinear uses no warmup

# Batch sizes match the AR(1) grid exactly (Table 3 in the paper).
# This creates a 15.8x step-count asymmetry between CD and CI, which is
# intentional and disclosed. The asymmetry FAVOURS CD (more gradient updates),
# so a null result under it is harder to dismiss than a matched-budget null.
# Changing these values would break the gamma=0 sanity check, which requires
# that this sweep's config be identical to the AR(1) grid at C=21, rho=0.5.
BATCH_BY_MODE: dict[str, int] = {"CI": 128, "CD": 8, "DLinear": 128}

N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1
print(f"N_PATCHES={N_PATCHES}  C*N={C * N_PATCHES}  total runs={len(GAMMAS)*len(SEEDS)*len(MODES)}")


In [ ]:
# ── Data generator (identical to train_leader_follower_v3) ────────────────────
N_LEADERS   = 10
N_FOLLOWERS = 10
N_ISOLATE   = 1
T_TOTAL     = 14_400
BURN_IN     = 1_000
TRAIN_FRAC  = 0.6
VAL_FRAC    = 0.2


def _make_transition(phi: float, gamma: float) -> np.ndarray:
    A = np.zeros((C, C), dtype=np.float64)
    np.fill_diagonal(A, phi)
    for k in range(N_FOLLOWERS):
        A[N_LEADERS + k, k] = gamma
    return A


def _make_cov(rho: float) -> np.ndarray:
    Sigma = np.full((C, C), rho, dtype=np.float64)
    np.fill_diagonal(Sigma, 1.0)
    if np.linalg.eigvalsh(Sigma).min() <= 0:
        raise ValueError(f"Covariance not PD for rho={rho}")
    return Sigma


def generate(gamma: float, rho: float, seed: int) -> np.ndarray:
    """VAR(1) leader-follower process. Returns (T_TOTAL, C) float64."""
    rng   = np.random.default_rng(seed)
    A     = _make_transition(PHI, gamma)
    L     = np.linalg.cholesky(_make_cov(rho))
    X     = np.zeros((T_TOTAL + BURN_IN, C), dtype=np.float64)
    for t in range(1, T_TOTAL + BURN_IN):
        X[t] = A @ X[t - 1] + L @ rng.standard_normal(C)
    return X[BURN_IN:]


def split_normalise(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_train = int(len(data) * TRAIN_FRAC)
    n_val   = int(len(data) * VAL_FRAC)
    train, val, test = data[:n_train], data[n_train:n_train+n_val], data[n_train+n_val:]
    mean = train.mean(axis=0, keepdims=True)
    std  = np.where(train.std(axis=0, keepdims=True) == 0, 1.0,
                    train.std(axis=0, keepdims=True))
    return (train-mean)/std, (val-mean)/std, (test-mean)/std


def make_windows(data: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    """Stride-tricks windowing — no Python loop."""
    T, Cv = data.shape
    n = T - SEQ_LEN - PRED_LEN + 1
    if n <= 0:
        raise ValueError(f"Not enough timesteps: {T}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(
        data, shape=(n, SEQ_LEN + PRED_LEN, Cv), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


# Quick window-count check
_n = int(T_TOTAL * TRAIN_FRAC) - SEQ_LEN - PRED_LEN + 1
print(f"Train windows: {_n}  "
      f"CI steps/epoch: {_n // 128 + (1 if _n % 128 else 0)}  "
      f"CD steps/epoch: {_n // 8   + (1 if _n % 8   else 0)}")


In [ ]:
# ── Model definitions ─────────────────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj    = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.proj(x))


class PatchTST_CI(nn.Module):
    """Channel-independent PatchTST. Input (B,L,C) -> (B,pred_len,C)."""

    def __init__(self) -> None:
        super().__init__()
        n = N_PATCHES
        self.embed   = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        layer        = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL*4,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head    = nn.Linear(n * D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        x  = x.permute(0, 2, 1).reshape(B * Cv, L)
        p  = x.unfold(-1, PATCH_SIZE, PATCH_STRIDE)           # (B*C, N, P)
        e  = self.encoder(self.embed(p))                       # (B*C, N, D)
        return self.head(e.reshape(B * Cv, -1)).reshape(B, Cv, -1).permute(0, 2, 1)


class PatchTST_CD(nn.Module):
    """Channel-dependent PatchTST with per-variate head. Input (B,L,C) -> (B,pred_len,C)."""

    def __init__(self) -> None:
        super().__init__()
        n = N_PATCHES
        self.embed   = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        layer        = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL*4,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        # Shared per-variate head: Linear(N*D, pred_len)
        self.head    = nn.Linear(n * D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        xp  = x.permute(0, 2, 1).reshape(B * Cv, L)
        p   = xp.unfold(-1, PATCH_SIZE, PATCH_STRIDE)         # (B*C, N, P)
        emb = self.embed(p).reshape(B, Cv, N_PATCHES, -1)     # (B, C, N, D)
        seq = emb.reshape(B, Cv * N_PATCHES, -1)              # (B, C*N, D)
        enc = self.encoder(seq).reshape(B * Cv, -1)           # (B*C, N*D)
        return self.head(enc).reshape(B, Cv, -1).permute(0, 2, 1)


class TrueDLinear(nn.Module):
    """DLinear (Zeng et al. 2023): MA trend + remainder, two independent Linear branches.
    Identical to train_dlinear_v2.ipynb."""

    def __init__(self) -> None:
        super().__init__()
        pad = (25 - 1) // 2   # kernel=25, symmetric padding -> output length == input
        self.avg_pool         = nn.AvgPool1d(kernel_size=25, stride=1, padding=pad)
        self.linear_trend     = nn.Linear(SEQ_LEN, PRED_LEN)
        self.linear_remainder = nn.Linear(SEQ_LEN, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        xf    = x.permute(0, 2, 1).reshape(B * Cv, 1, L)     # (B*C, 1, L)
        trend = self.avg_pool(xf).reshape(B * Cv, L)[:, :L]   # (B*C, L)
        rem   = xf.reshape(B * Cv, L) - trend
        out   = self.linear_trend(trend) + self.linear_remainder(rem)
        return out.reshape(B, Cv, -1).permute(0, 2, 1)


def build_model(mode: str) -> nn.Module:
    if mode == "CI":      return PatchTST_CI()
    if mode == "CD":      return PatchTST_CD()
    if mode == "DLinear": return TrueDLinear()
    raise ValueError(f"Unknown mode: {mode}")


# ── Architecture assertions ───────────────────────────────────────────────────
_cd = build_model("CD")
assert _cd.head.in_features  == N_PATCHES * D_MODEL, f"CD head wrong: {_cd.head}"
assert _cd.head.out_features == PRED_LEN,             f"CD head wrong: {_cd.head}"
print(f"CD head OK: {_cd.head}")
del _cd

_dl = build_model("DLinear")
assert _dl.linear_trend is not _dl.linear_remainder, "DLinear branches must be independent"
_x  = torch.zeros(2, SEQ_LEN, C)
assert _dl(_x).shape == (2, PRED_LEN, C)
print(f"TrueDLinear OK: AvgPool1d(k=25,pad=12), two independent Linear({SEQ_LEN},{PRED_LEN})")
del _dl, _x
free_cuda()


In [ ]:
# ── Training engine ───────────────────────────────────────────────────────────

def _cosine_warmup(optimizer, epoch: int, warmup: int) -> None:
    if epoch < warmup:
        lr = LR * (epoch + 1) / warmup
    else:
        progress = (epoch - warmup) / max(1, MAX_EPOCHS - warmup)
        lr = LR * 0.5 * (1.0 + np.cos(np.pi * progress))
    for g in optimizer.param_groups:
        g["lr"] = lr


@torch.no_grad()
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE)).cpu()
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred,  yb, reduction="sum").item()
        n   += yb.numel()
    return mse / n, mae / n


def _fit(mode: str, gamma: float, seed: int,
         datasets: tuple, batch_size: int) -> dict:
    x_tr, y_tr, x_va, y_va, x_te, y_te = datasets
    train_dl = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size,
                          shuffle=True,  drop_last=False)
    val_dl   = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size,
                          shuffle=False, drop_last=False)
    test_dl  = DataLoader(TensorDataset(x_te, y_te), batch_size=batch_size,
                          shuffle=False, drop_last=False)

    use_warmup = mode in ("CI", "CD")
    model      = build_model(mode).to(DEVICE)
    opt        = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler     = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion  = nn.MSELoss()

    spe   = len(train_dl)   # steps per epoch
    best_val, best_epoch, best_steps = float("inf"), 0, 0
    best_state, no_improve, total    = None, 0, 0

    try:
        for epoch in range(MAX_EPOCHS):
            if use_warmup:
                _cosine_warmup(opt, epoch, WARMUP_EPOCHS)
            model.train()
            for xb, yb in train_dl:
                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                total += 1

            val_mse, _ = _evaluate(model, val_dl)
            if val_mse < best_val:
                best_val   = val_mse
                best_epoch = epoch + 1
                best_steps = total
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
    finally:
        if best_state:
            model.load_state_dict(best_state)
        test_mse, test_mae = _evaluate(model, test_dl)
        del model, opt, scaler
        free_cuda()

    return {"dataset": "leader_follower_var1", "C": C, "rho": RHO, "gamma": gamma,
            "mode": mode, "seed": seed, "test_mse": test_mse, "test_mae": test_mae,
            "best_epoch": best_epoch, "batch_size": batch_size,
            "steps_per_epoch": spe, "total_steps": best_steps}


def train_one(mode: str, gamma: float, seed: int) -> dict:
    """Generate data once, then fit with OOM-safe batch halving."""
    set_seed(seed)
    raw = generate(gamma, RHO, seed)
    tr, va, te = split_normalise(raw)
    datasets   = (*make_windows(tr), *make_windows(va), *make_windows(te))
    batch_size = BATCH_BY_MODE[mode]

    while True:
        try:
            set_seed(seed)   # identical init on every OOM retry
            return _fit(mode, gamma, seed, datasets, batch_size)
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            free_cuda()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f"  OOM: retrying at batch_size={batch_size}")


In [ ]:
# ── Main sweep with atomic checkpointing and safe resume ─────────────────────
OUT   = Path("/kaggle/working/results_leader_follower.csv")
TMP   = OUT.with_suffix(".csv.tmp")


def _key(gamma: float, mode: str, seed: int) -> tuple:
    return (round(float(gamma), 4), str(mode), int(seed))


def _save(rows: list[dict]) -> None:
    pd.DataFrame(rows).to_csv(TMP, index=False)
    os.replace(TMP, OUT)


if OUT.exists() and OUT.stat().st_size > 100:
    _existing = pd.read_csv(OUT)
    done      = {_key(r.gamma, r.mode, r.seed) for r in _existing.itertuples()}
    results   = _existing.to_dict("records")
    print(f"Resuming: {len(done)}/{len(GAMMAS)*len(MODES)*len(SEEDS)} runs done.")
else:
    done, results = set(), []

total = len(GAMMAS) * len(MODES) * len(SEEDS)
idx   = 0
fails = []

for gamma in GAMMAS:
    for mode in MODES:
        for seed in SEEDS:
            idx += 1
            key = _key(gamma, mode, seed)
            if key in done:
                print(f"[{idx}/{total}] SKIP gamma={gamma} mode={mode} seed={seed}")
                continue
            print(f"[{idx}/{total}] gamma={gamma} mode={mode} seed={seed} ...",
                  end=" ", flush=True)
            t0 = time.time()
            try:
                row = train_one(mode, gamma, seed)
            except Exception as exc:
                free_cuda()
                print(f"FAILED: {type(exc).__name__}: {exc}")
                fails.append((gamma, mode, seed, repr(exc)))
                continue
            print(f"mse={row['test_mse']:.4f}  epoch={row['best_epoch']}"
                  f"  bs={row['batch_size']}  ({time.time()-t0:.0f}s)")
            results.append(row)
            done.add(key)
            _save(results)

print(f"\nDone. {len(results)}/{total} runs -> {OUT}")
if fails:
    print(f"{len(fails)} failures (rerun this cell to retry):")
    for f in fails:
        print(" ", f)


In [ ]:
# ── Summary and Table 7 reconstruction ───────────────────────────────────────
from pathlib import Path

df = pd.read_csv(OUT)
print(f"Rows: {len(df)}  |  seeds: {sorted(df['seed'].unique())}  "
      f"|  modes: {sorted(df['mode'].unique())}")
print(f"Duplicates: {df.duplicated(['gamma','mode','seed']).sum()}")

# Full Table 7 reconstruction
piv = df.groupby(["gamma","mode"])["test_mse"].agg(["mean","std"]).unstack("mode")
piv.columns = [f"{stat}_{mode}" for stat,mode in piv.columns]

print("\n=== Table 7 reconstruction (mean ± std over seeds) ===")
print(f"{'gamma':>6} {'CI':>8} {'CD':>8} {'DLinear':>8} {'CD/CI':>7} {'DL-CI':>7}")
for g in sorted(df['gamma'].unique()):
    ci  = piv.loc[g,'mean_CI']
    cd  = piv.loc[g,'mean_CD']
    dl  = piv.loc[g,'mean_DLinear']
    print(f"{g:>6.1f} {ci:>8.4f} {cd:>8.4f} {dl:>8.4f} "
          f"{cd/ci:>7.4f} {dl-ci:>+7.4f}")

# DLinear trend with gamma
print("\n=== DLinear MSE by gamma (should decrease slightly as gamma increases) ===")
print(df[df['mode']=='DLinear'].groupby('gamma')['test_mse'].mean().round(4).to_string())

# gamma=0 sanity check vs AR(1) grid
print("\n=== gamma=0.0 sanity check ===")
g0 = df[np.isclose(df['gamma'], 0.0) & df['mode'].isin(['CI','CD'])]
print("Unified sweep gamma=0 means:")
print(g0.groupby('mode')['test_mse'].mean().round(4).to_string())

for cand in ["/kaggle/input/previous_results/results_grid_canonical.csv",
             "/kaggle/working/results_grid_canonical.csv"]:
    if Path(cand).exists():
        grid = pd.read_csv(cand)
        ref  = grid[(grid.C==21) & np.isclose(grid.rho, 0.5)]
        print("AR(1) grid C=21 rho=0.5 means:")
        print(ref.groupby('mode')['test_mse'].mean().round(4).to_string())
        diff = (g0.groupby('mode')['test_mse'].mean()
                - ref.groupby('mode')['test_mse'].mean())
        print(f"Diff (unified - grid): CI={diff.get('CI',float('nan')):.4f}  "
              f"CD={diff.get('CD',float('nan')):.4f}  (should be < 0.02)")
        break


In [ ]:
from IPython.display import FileLink
FileLink(str(OUT))
